# Rule of Thumb — tabular quickstart

This notebook demonstrates the tabular RoT explainer on a small synthetic
problem: we define a toy "black box" classifier, fit a Rule-of-Thumb
surrogate to its outputs, and inspect the learned feature importances.

Everything runs on CPU with synthetic data in a few seconds.

In [1]:
import numpy as np

import ruleofthumb

In [2]:
# Synthetic data and a trivially simple "black box"
rng = np.random.RandomState(0)
n, d = 2000, 5
X = rng.randn(n, d).astype(np.float32)

def black_box(X):
    # logistic rule driven by features 0 and 2 only
    z = 2.0 * X[:, 0] - 1.5 * X[:, 2]
    return (1 / (1 + np.exp(-z)) > 0.5).astype(np.int64)  # int labels

y = black_box(X)

In [3]:
rot = ruleofthumb.fit(y_outputs=y, x_inputs=X, epochs=50, batch_size=500, learning_rate=0.05)
importances = rot.get_explanation(X)  # signed: positive = evidence toward class 1
importances[:3]

array([[ 2.6098304e+00,  3.4245197e-03, -9.5335674e-01, -1.6586026e-02,
        -9.2830844e-02],
       [-1.3690014e+00,  1.2082413e-02,  9.0174928e-02,  2.3185296e-03,
        -3.0240336e-02],
       [ 2.5851119e-01,  2.0020097e-02, -7.5233197e-01,  5.0482841e-04,
        -3.1669378e-02]], dtype=float32)

In [4]:
# Global feature ranking: mean |importance| per feature (a magnitude view
# for ranking only, like SHAP summary plots — the explanations themselves
# are signed). Features 0 and 2 should dominate.
mean_imp = np.abs(importances).mean(axis=0)
for i, v in enumerate(mean_imp):
    print(f"feature {i}: {v:.4f}")

feature 0: 1.1674
feature 1: 0.0123
feature 2: 0.7348
feature 3: 0.0066
feature 4: 0.0349


## Fidelity vs. number of revealed features

`score_ordering` reveals features most-important-first and measures how well
the partially-revealed RoT reproduces the black box's labels.

In [5]:
import torch

x_t = torch.from_numpy(X)
y_t = torch.from_numpy(y.astype(np.int64))
order = rot.get_order(x_t)
acc = rot.score_ordering(x_t, y_t, order)
print("accuracy after revealing k=1..d features:")
print(acc.numpy())

accuracy after revealing k=1..d features:
[0.5035 0.9815 0.982  0.978  0.978  0.977 ]


## Multiclass

The same pipeline works for any number of classes via `n_classes=`:
explanations come back per class (`[N, n_classes, d]`, signed), and
`score_ordering` defaults to plain accuracy, with an optional per-step
K×K confusion matrix via `return_confusion=True`.

In [6]:
# A 3-class black box driven by features 0 and 2
y3 = (X[:, 0] > 0.5).astype(np.int64) + (X[:, 2] > -0.5).astype(np.int64)  # labels in {0, 1, 2}

rot3 = ruleofthumb.fit(y_outputs=y3, x_inputs=X, epochs=50, batch_size=500, learning_rate=0.05, n_classes=3)
exp3 = rot3.get_explanation(X)
print("per-class explanations:", exp3.shape)  # (N, 3, d), signed

order3 = rot3.get_order(x_t)
acc3 = rot3.score_ordering(torch.from_numpy(X), torch.from_numpy(y3), order3)
print("multiclass accuracy curve:", acc3.numpy())

confusion3 = rot3.score_ordering(
    torch.from_numpy(X), torch.from_numpy(y3), order3, return_confusion=True
)
print("confusion after revealing all features (rows = true label):")
print(confusion3[-1].numpy())

per-class explanations: (2000, 3, 5)
multiclass accuracy curve: [0.5745 0.646  0.795  0.7945 0.7955 0.795 ]
confusion after revealing all features (rows = true label):
[[ 282  164    0]
 [  64 1025   60]
 [   0  122  283]]
